# Run one WP9 simulation shard

Launch this notebook in 27 Colab sessions. Set a distinct `SHARD_INDEX` from 0 through 26 in each session. Each shard receives a deterministic subset of the 480 WP9 cells and writes to its own JSON checkpoint.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = os.environ.get("WDCF_REPO_URL", "https://github.com/hugogobato/wasserstein-causal-forests.git")
REPO_DIR = Path("/content/wasserstein-causal-forests")
if not (REPO_DIR / "research/sim/runner.py").exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
sys.path.insert(0, str(REPO_DIR / "research"))
os.chdir(REPO_DIR)
print("Repository:", REPO_DIR)

In [ ]:
# Change only SHARD_INDEX between sessions.
SHARD_INDEX = int(os.environ.get("WDCF_SHARD_INDEX", "0"))
NUM_SHARDS = 27
DGPS = ("D0", "D1", "D2", "D3", "D4", "D5", "D8")
N_REGIONS = (500, 1000)
N_SEEDS = 30
N_TREES = 200
N_EVAL = 200
WORKERS = 1  # Keep one process per Colab session; parallelism is across sessions.
CLAIM_ID = "WP9-T3-colab"
USE_DRIVE = False
RESUME = True

if not 0 <= SHARD_INDEX < NUM_SHARDS:
    raise ValueError("SHARD_INDEX must be between 0 and 26")
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_DIR = Path("/content/drive/MyDrive/wasserstein-causal-forests/wp9_shards")
else:
    OUTPUT_DIR = Path("/content/wp9_shards")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH = OUTPUT_DIR / f"wp9_shard_{SHARD_INDEX:02d}.json"
print("Shard", SHARD_INDEX, "of", NUM_SHARDS, "->", OUTPUT_PATH)

In [ ]:
from sim.runner import build_simulation_tasks
tasks = build_simulation_tasks(
    dgp_names=DGPS, n_regions_list=N_REGIONS, n_seeds=N_SEEDS,
    n_trees=N_TREES, n_eval=N_EVAL, shard_index=SHARD_INDEX,
    num_shards=NUM_SHARDS, claim_id=CLAIM_ID,
)
print(f"This session owns {len(tasks)} cells")
print("First cell:", tasks[0][:4], "last cell:", tasks[-1][:4])

In [ ]:
command = [
    sys.executable, "research/sim/runner.py",
    "--dgps", *DGPS, "--n", *(str(n) for n in N_REGIONS),
    "--seeds", str(N_SEEDS), "--n_trees", str(N_TREES),
    "--n_eval", str(N_EVAL), "--workers", str(WORKERS),
    "--shard-index", str(SHARD_INDEX), "--num-shards", str(NUM_SHARDS),
    "--claim", CLAIM_ID, "--out", str(OUTPUT_PATH),
]
if RESUME:
    command.append("--resume")
print("Running:", " ".join(command))
subprocess.run(command, check=True)

In [ ]:
import json
rows = json.loads(OUTPUT_PATH.read_text())
cells = {(r["dgp_id"], r["n_regions"], r["observation_regime"], r["seed"]) for r in rows}
print(f"Saved {len(rows)} rows covering {len(cells)} cells")

# Safe download fallback for non-Drive Colab sessions.
try:
    from google.colab import files
    files.download(str(OUTPUT_PATH))
    print("Downloaded:", OUTPUT_PATH)
except Exception as e:
    print("(Not on Colab / download skipped):", e)